## Domo AI Pro Operations & Processing
Domo AI Pro operations and processing will consume credits as described on [Domo's online consumption terms](https://www.domo.com/consumption-terms). Please see the credit rate card in your Domo instance for more information. (Admin > Company Settings > Credit Utilization > Rate Card)

# Chat Completion with domojupyter AI

This notebook demonstrates how to use `ai.chat_completion()` to build conversational interactions
with a language model — including multi-turn conversations — from within your Domo Jupyter workspace.

**Function signature:**
```python
ai.chat_completion(
    input_str,              # required: the current user message
    chat_context=None,      # optional: list of {"role": "user"|"assistant", "content": str} prior messages
    system=None,            # optional: system message string
    model=None,             # optional: model ID
    temperature=None,       # optional: float
    max_tokens=None,        # optional: int
    reasoning_config=None,  # optional: dict e.g. {"enabled": True, "budgetTokens": 1000}
    response_format=None    # optional: dict e.g. {"type": "JSON", "schema": {...}}
)
# Returns: TextAIResponse with .choices[0]
```

In [ ]:
import domojupyter.ai as ai
import json

## 1. Single-Turn Chat

The simplest usage — ask a single question and receive a response.
Ideal for one-off analytical queries against business context.

In [ ]:
response = ai.chat_completion(
    "What does a net revenue retention rate above 100% indicate about a SaaS business?"
)

print(response.choices[0])

## 2. Multi-Turn Conversation (Continuing a Chat)

Use `chat_context` to pass prior exchanges so the model maintains context across turns.
Each entry in the list is a dict with `role` (`"user"` or `"assistant"`) and `content`.

This enables iterative analysis workflows where each follow-up question builds on the previous answer.

In [ ]:
# Turn 1: initial question
turn1_response = ai.chat_completion(
    "Our average deal size dropped 18% this quarter while win rate stayed flat. What are the likely causes?"
)
turn1_answer = turn1_response.choices[0]
print("Turn 1:")
print(turn1_answer)
print()

In [ ]:
# Turn 2: follow-up question, passing prior context
chat_context = [
    {"role": "user", "content": "Our average deal size dropped 18% this quarter while win rate stayed flat. What are the likely causes?"},
    {"role": "assistant", "content": turn1_answer}
]

turn2_response = ai.chat_completion(
    "Which of those causes can be diagnosed using CRM data, and what fields or reports should I look at?",
    chat_context=chat_context
)
turn2_answer = turn2_response.choices[0]
print("Turn 2:")
print(turn2_answer)
print()

In [ ]:
# Turn 3: continue the conversation with full history
chat_context.append({"role": "user", "content": "Which of those causes can be diagnosed using CRM data, and what fields or reports should I look at?"})
chat_context.append({"role": "assistant", "content": turn2_answer})

turn3_response = ai.chat_completion(
    "Write a SQL query that would help surface deals below our historical average deal size that closed this quarter.",
    chat_context=chat_context
)
print("Turn 3:")
print(turn3_response.choices[0])

## 3. With a System Persona

Use the `system` parameter to assign the model a specific role or expertise area.
This shapes how it frames answers and what context it prioritizes in its responses.

In [ ]:
response = ai.chat_completion(
    "We have 90 days to reduce operating expenses by 15% without cutting headcount. Where should we start?",
    system=(
        "You are a CFO advisor specializing in SaaS company cost optimization. "
        "You focus on vendor consolidation, tooling rationalization, and process efficiency. "
        "Always prioritize actions that preserve revenue-generating capacity."
    )
)

print(response.choices[0])

## 4. Structured JSON Response

Use `response_format` to request a structured JSON response from the model.
This allows you to programmatically consume the output — for example, to write results to a Domo dataset or feed downstream logic.

In [ ]:
response_schema = {
    "type": "JSON",
    "schema": {
        "properties": {
            "risk_level": {"type": "string"},
            "primary_concern": {"type": "string"},
            "recommended_action": {"type": "string"},
            "urgency_score": {"type": "number"}
        },
        "required": ["risk_level", "primary_concern", "recommended_action", "urgency_score"]
    }
}

response = ai.chat_completion(
    (
        "Evaluate the following business situation and return a structured risk assessment: "
        "A mid-market SaaS company has 3 customers each representing over 15% of ARR, "
        "two of which are up for renewal in the next 60 days with no QBR completed."
    ),
    system="You are a business risk analyst. Return only a structured JSON assessment.",
    response_format=response_schema
)

raw = response.choices[0]
assessment = json.loads(raw)

print(f"Risk Level:          {assessment['risk_level']}")
print(f"Primary Concern:     {assessment['primary_concern']}")
print(f"Recommended Action:  {assessment['recommended_action']}")
print(f"Urgency Score:       {assessment['urgency_score']:.1f}")